# Minimal Segmentation
- This allows you to segment vasculature and explore the t0, t2 signal intensities and the segmented vasculature
- You can take images and make movies of key frames
- Run this on the nap-ij-record kernel

In [9]:
import numpy as np
from skimage.transform import rescale
from skimage.measure import label, regionprops_table
from skimage import util
import napari
from liffile import LifFile
from pathlib import Path
import imagej
from imagej import Mode
import scyjava as sj
from matplotlib.colors import to_rgba
from magicgui import magicgui
from magicgui.widgets import TextEdit, Container, create_widget
from typing import Optional
import os
import traceback
from napari.utils.notifications import show_error, show_info, show_warning


# import warnings
# warnings.filterwarnings("ignore")

ij = imagej.init("sc.fiji:fiji", mode=Mode.INTERACTIVE, add_legacy=True)

IJ = sj.jimport("ij.IJ")
Duplicator = sj.jimport("ij.plugin.Duplicator")
WekaSegmentation = sj.jimport("trainableSegmentation.WekaSegmentation")
ImagePlus = sj.jimport("ij.ImagePlus")
ImageStack = sj.jimport("ij.ImageStack")
FloatProcessor = sj.jimport("ij.process.FloatProcessor")

In [2]:
def numpy_to_imageplus(arr: np.ndarray, title="image") -> ImagePlus:
    if arr.ndim == 2:
        y, x = arr.shape
        pix = np.asarray(arr, dtype=np.float32).ravel()
        java_floats = sj.jarray("f", pix.size)
        # fill java float[]
        for i, v in enumerate(pix.tolist()):
            java_floats[i] = float(v)
        fp = FloatProcessor(x, y, java_floats)
        return ImagePlus(title, fp)

    if arr.ndim == 3:
        z, y, x = arr.shape
        stack = ImageStack(x, y)
        for zi in range(z):
            pix = np.asarray(arr[zi], dtype=np.float32).ravel()
            java_floats = sj.jarray("f", pix.size)
            for i, v in enumerate(pix.tolist()):
                java_floats[i] = float(v)
            fp = FloatProcessor(x, y, java_floats)
            stack.addSlice(fp)
        return ImagePlus(title, stack)

    raise ValueError(f"Unsupported shape: {arr.shape}")

In [3]:
def imageplus_to_numpy(imp: ImagePlus) -> np.ndarray:
    w, h = imp.getWidth(), imp.getHeight()
    n = imp.getStackSize()
    stack = imp.getStack()
    out = []
    for z in range(1, n + 1):
        ip = stack.getProcessor(z)
        pix = np.array(ip.getPixels()).reshape(h, w)
        out.append(pix)
    return np.stack(out, axis=0) if n > 1 else out[0]

In [4]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [5]:
def apply_weka_with_exact_preprocessing(image: np.ndarray, model_path: str) -> np.ndarray:
    imp = numpy_to_imageplus(image, title="image")
    dup = Duplicator().run(imp)
    IJ.run(dup, "8-bit", "")
    IJ.run(dup, "Auto Threshold", "method=Otsu stack")
    IJ.run(dup, "Erode (3D)", "iso=255")

    try:
        seg = WekaSegmentation(dup)
        seg.loadClassifier(model_path)
        out_imp = seg.applyClassifier(dup, 0, False)
        if out_imp is None:
            out_imp = seg.getClassifiedImage()

        if out_imp is None:
            raise RuntimeError("No classified image returned (out_imp is None). Likely a Java-side error or model/input mismatch.")

        return imageplus_to_numpy(out_imp)

    except:
        # Show Java exception details if present
        print("=== Python/Java exception ===")

In [6]:
def step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [7]:
def segment_to_view(i, lif, lif_path, classifier_path):
    img = lif.images[i]
    filename = (
        os.path.basename(lif_path).lower().replace(".lif", "")
        + "__"
        + "".join(img.path)
    )

    show_info(f"Now segmenting: {filename}")

    if img.dims != ("T", "Z", "Y", "X"):
        # treat as user-facing issue, not a silent failure
        raise ValueError(f"Wrong dimensions {img.dims}; expected ('T','Z','Y','X').")

    image = img.asarray()
    xa = img.asxarray()

    # liffile coordinates are typically in meters → convert to µm
    x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
    z_um = step("Z", xa) * 1e6 if step("Z", xa) is not None else None

    if x_um is None or z_um is None or x_um == 0:
        raise ValueError(f"Missing/invalid pixel size metadata (x_um={x_um}, z_um={z_um}).")

    try:
        t0 = image[0, :, :, :]
        t2 = image[2, :, :, :]

        gel_matrix = apply_weka_with_exact_preprocessing(t0, classifier_path)

        vasculature_segmentation = (gel_matrix == 0).astype(int)
        vasculature_labels = label(vasculature_segmentation)

        table = regionprops_table(vasculature_labels, properties=("label", "area"))
        condition = (table["area"] >= 20)
        input_labels = table["label"]
        output_labels = input_labels * condition
        output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
        clean_vasculature_segmentation = output_labels > 0

        t2 = convert(t2, 0, 255, np.uint8)
        t0 = convert(t0, 0, 255, np.uint8)

        scale = (z_um / x_um, 1, 1)
        rescaled_t0 = rescale(scale=scale, image=t0, anti_aliasing=False)
        rescaled_t2 = rescale(scale=scale, image=t2, anti_aliasing=False)
        rescaled_vasculature_segmentation = rescale(
            scale=scale,
            image=clean_vasculature_segmentation,
            anti_aliasing=False,
            order=0,
            preserve_range=True,
        ).astype(clean_vasculature_segmentation.dtype)

        return rescaled_t0, rescaled_t2, rescaled_vasculature_segmentation

    except Exception as e:
        # show a concise message to the user
        show_error(f"Segmentation failed for {filename}: {type(e).__name__}: {e}")
        # (optional) print full traceback to console for debugging
        traceback.print_exc()
        raise  # re-raise so caller can decide what to do
    finally:
        # Ensure cleanup always happens
        try:
            IJ.run("Close All")
        except Exception:
            pass

In [10]:
viewer = napari.Viewer()

_selected_lif_path: Optional[Path] = None

images_output = TextEdit(value="")
try:
    images_output.native.setReadOnly(True)
except Exception:
    pass
images_output.min_height = 120
images_output.max_height = 300


@magicgui(
    lif_path={"label": "Select .lif", "mode": "r", "filter": "*.lif"},
    call_button="List images",
)
def list_images(lif_path: Path = Path()):
    global _selected_lif_path

    if not lif_path or not lif_path.exists():
        images_output.value = "[WARN] Please select a .lif file."
        return

    _selected_lif_path = lif_path
    images_output.value = f"[INFO] Selected: {lif_path}\n[INFO] Reading images…"

    lines = []
    try:
        with LifFile(lif_path) as lif:
            for i, img in enumerate(lif.images):
                name = "".join(getattr(img, "path", ()))
                lines.append(f"{i}\t{name}")
    except Exception as e:
        images_output.value = (
            images_output.value
            + f"\n[ERROR] Failed to read .lif: {type(e).__name__}: {e}"
        )
        return

    if not lines:
        images_output.value = images_output.value + "\n[WARN] No images found."
        return

    images_output.value = (
        images_output.value
        + f"\n[OK] Found {len(lines)} images:\n\n"
        + "\n".join(lines)
    )


@magicgui(
    image_index={"label": "Image index", "min": 0, "step": 1},
    classifier_path={"label": "Select classifier", "mode": "r", "filter": "*.model"},
    clear_layers={"label": "Clear viewer first"},
    call_button="Segment + View",
)
def segment_and_view(
    image_index: int = 0,
    classifier_path: Path = Path(),
    clear_layers: bool = True,
):
    global _selected_lif_path

    if _selected_lif_path is None or not _selected_lif_path.exists():
        images_output.value = (
            images_output.value.rstrip()
            + "\n[WARN] Select a .lif file and click ‘List images’ first."
            if images_output.value
            else "[WARN] Select a .lif file and click ‘List images’ first."
        )
        return

    if not classifier_path or not classifier_path.exists():
        images_output.value = (
            images_output.value.rstrip()
            + "\n[WARN] Select a classifier .model file."
            if images_output.value
            else "[WARN] Select a classifier .model file."
        )
        return

    images_output.value = (
        images_output.value.rstrip()
        + f"\n[INFO] Segmenting image_index={image_index} with classifier={classifier_path}…\n"
        if images_output.value
        else f"[INFO] Segmenting image_index={image_index} with classifier={classifier_path}…\n"
    )

    try:
        with LifFile(_selected_lif_path) as lif:
            n = len(lif.images)
            if image_index < 0 or image_index >= n:
                images_output.value = images_output.value.rstrip() + f"\n[WARN] Image index out of range (0..{n-1})."
                return

            t0, t2, segmentation = segment_to_view(
                image_index,
                lif,
                str(_selected_lif_path),
                str(classifier_path),
            )
    except Exception as e:
        images_output.value = images_output.value.rstrip() + f"\n[ERROR] Segmentation failed: {type(e).__name__}: {e}"
        return

    if clear_layers:
        viewer.layers.clear()
        images_output.value = images_output.value.rstrip() + "\n[INFO] Cleared viewer layers."

    viewer.add_image(t0, name="t0")
    viewer.add_image(t2, name="t2")
    images_output.value = images_output.value.rstrip() + "\n[INFO] Added t0 and t2 layers."

    labels = viewer.add_labels(segmentation.astype("int32"), name="vasculature")
    labels.color_mode = "direct"
    images_output.value = images_output.value.rstrip() + "\n[OK] Added vasculature labels (color_mode=direct)."

    images_output.value = images_output.value.rstrip() + "\n[OK] Segmentation complete."


list_panel = Container(widgets=[list_images, images_output])
viewer.window.add_dock_widget(list_panel, area="right")
viewer.window.add_dock_widget(segment_and_view, area="right")


INFO: Now segmenting: m4_2024.10.24_fl2_fl6__rLN2_24.10.24_device1/P 2
